# 52. Models (baseline decide TF-IDF & SVD)
## Contents
- Prerequisites
- Baseline model (decide TF-IDF & SVD paramters)
---------------------------------------------------------
## Prerequisites

In [5]:
import time
import os
import pandas as pd
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from tqdm import tqdm
import joblib
import warnings
from sklearn.model_selection import cross_val_score
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
file_path = 'data/'
parties = np.load('00_parties.npy')
options = ['imbalanced','oversampling','undersampling','balancedsampling']
rs = 1

------------------------------------------------
<br>
<br>
<br>


## Baseline models (decide TF-IDF & SVD)
--------------------------------------------

In [6]:
dur = time.time()
options = ['imbalanced']
results = pd.DataFrame(columns=['party', 'option', 'bm_score', 'lr_score', 'rf_score', 'svm_score'])
# -------------------------------------------------------------------------------------~----------------------------------------
for party in tqdm(parties, desc = 'Party loop'):                                       # for each party
    for option in options:                                                             # for each sampling method

# load training data 
# -------------------------------------------------------------------------------------~----------------------------------------
        X_train_url = file_path + f"20_models/{option}/" + f"{party}_X_train.csv"      # define X_train url
        y_train_url = file_path + f"20_models/{option}/" + f"{party}_y_train.csv"      # define y_train url
        X_train = pd.read_csv(X_train_url)                                             # read X_train
        X_train = X_train.drop(columns=['source', 'text','stemming_id','document_id']) # redefine X_train without columns
        y_train = pd.read_csv(y_train_url)                                             # read y_train

# load validation data
# -------------------------------------------------------------------------------------~----------------------------------------      
        X_valid_url = file_path + f"20_models/{option}/" + f"{party}_X_valid.csv"      # define X_valid url
        y_valid_url = file_path + f"20_models/{option}/" + f"{party}_y_valid.csv"      # define y_valid url
        X_valid = pd.read_csv(X_valid_url)                                             # read X_valid
        X_valid = X_valid.drop(columns=['source', 'text','stemming_id','document_id']) # redefine X-valid without columns
        y_valid = pd.read_csv(y_valid_url)                                             # read y_valid

# models
# -------------------------------------------------------------------------------------~----------------------------------------
        # baseline model
        # ----------------------------------------------------------------
        bm = DummyClassifier(strategy='most_frequent')                                 # define model, dummy (most frequent)
        bm.fit(X_train, y_train)                                                       # fit model on X_train and y_train
        bm_score = bm.score(X_valid, y_valid)                                          # score model on X_valid and y_valid

        # logistic regression model
        # ----------------------------------------------------------------
        lr = LogisticRegression(random_state = rs)                                     # define model, logistic regression
        lr.fit(X_train, y_train)                                                       # fit model on X_train and y_train
        lr_score = lr.score(X_valid, y_valid)                                          # score model on X_valid and y_valid

        # random forest model
        # ----------------------------------------------------------------
        rf = RandomForestClassifier(random_state = rs)                                 # define model, random forest
        rf.fit(X_train, y_train)                                                       # fit model on X_train and y_train
        rf_score = rf.score(X_valid, y_valid)                                          # score model on X_valid and y_valid

        # support vector machine model
        # ----------------------------------------------------------------
        svm = SVC(random_state = rs)                                                   # define model, support vector machine
        svm.fit(X_train, y_train)                                                      # fit model on X_train and y_train
        svm_score = svm.score(X_valid, y_valid)                                        # score model on X_valid and y_valid

        # define results
        # ----------------------------------------------------------------
        result = pd.DataFrame({
            'party': [party],
            'option': [option],
            'bm_score': [bm_score],
            'lr_score': [lr_score],
            'rf_score': [rf_score],
            'svm_score': [svm_score]})                                                 # define result
        results = pd.concat([results, result], ignore_index=True)                      # redefine results
        #display(result)

# -------------------------------------------------------------------------------------~----------------------------------------
display(results)                                                                       # display results
print('\n---------------------------------------------------------------------------------------------------------------------')
print(f"Code duration: {round((time.time()  - dur),3)} seconds")    

Party loop: 100%|██████████| 18/18 [15:42<00:00, 52.35s/it]


,party,option,bm_score,lr_score,rf_score,svm_score
0,50PLUS,imbalanced,0.790295,0.795246,0.799703,0.790295
1,CDA,imbalanced,0.502665,0.671754,0.670543,0.532461
2,CU,imbalanced,0.538331,0.682436,0.680980,0.562834
3,D66,imbalanced,0.527630,0.679593,0.679835,0.552109
4,DENK,imbalanced,0.824800,0.834264,0.840087,0.824800
5,FVD,imbalanced,0.605934,0.688328,0.711869,0.605934
6,GLPvdA,imbalanced,0.801839,0.817324,0.826276,0.801839
7,PVV,imbalanced,0.614788,0.695758,0.716606,0.614788
8,PvdD,imbalanced,0.784085,0.793062,0.806162,0.784085
9,SGP,imbalanced,0.636077,0.680630,0.695400,0.636077



---------------------------------------------------------------------------------------------------------------------
Code duration: 942.28 seconds


In [4]:
numeric_columns = results.select_dtypes(include=[np.number]).columns
average_results = results.groupby('option')[numeric_columns].mean().reset_index()
print(average_results)

       option  bm_score  lr_score  rf_score  svm_score
0  imbalanced  0.687095  0.742669  0.750424   0.691418


-------------------------------------------------

| option       | bm_score | lr_score | rf_score | svm_score | ngram | minocc_ratio | ncomp_ratio |
|:-------------|:--------:|:--------:|:--------:|:---------:|:-----:|:------------:|:-----------:|
| imbalanced   | 0.687743 | 0.736021 | 0.745534 | 0.690181  | 1.3   |    0.002     |    0.04     |
| imbalanced   | 0.687743 | 0.737212 | 0.746244 | 0.690194  | 1.2   |    0.002     |    0.04     |
| imbalanced   | 0.687743 | 0.730627 | 0.741954 | 0.690181  | 1.2   |    0.003     |    0.04     |
| imbalanced   | 0.687743 | 0.744476 | 0.747244 | 0.69034   | 1.2   |    0.001     |    0.04     |
| imbalanced   | 0.687095 | 0.742669 | 0.750424 | 0.691418  | 1.2   |    0.001     |    0.03     | X
| imbalanced   | 0.687743 | 0.734658 | 0.748233 | 0.690315  | 1.2   |    0.001     |    0.02     |
| imbalanced   | 0.687743 | 0.738203 | 0.748149 | 0.690261  | 1.3   |    0.001     |    0.03     |
| imbalanced   | 0.687743 | 0.731488 | 0.745857 | 0.690221  | 1.2   |    0.002     |    0.03     |

--> The best parameters are 1.2 (bigrams), minocc of 0.01 and 0.03 components